# ArraySplitter Methods Application Note — full reproducer (v2 preprint)

**Frozen:** 2026-05-22  
**Servers:** aglab0.utrail.org (compute), Theseus 65.109.35.157 (regression-test)  
**Paper:** outputs/papers/asplit-optimization/drafts/paper-draft-2026-05-21-v2.md  
**ART:** ART-paper-asplit-tool-2026-05-21-v2  

This notebook reproduces every numerical claim in the v2 preprint:

- §3.1 — four-way comparison on the zebra finch panel (ArraySplitter, FasTAN, centroAnno, TRASH)
- §3.2 — four-way comparison on the HG002.mat panel
- §3.3 — biological findings on zfinch (3 monomer classes; 88 HOR arrays; 191 → 10–13 bp recursive architecture on 8 chromosomes; chrW 47× HOR)
- §3.4 — rotation problem (period-aware cut Jaccard between ArraySplitter and FasTAN)
- §3.5 — max-period cap sweep (D = 5 000 / 32 000 / 100 000 / 250 000) on both panels
- §3.6 — nested HOR matryoshka (period_classes histogram across recursion levels)

**Comparator commit pins:** FasTAN ad3002/963ac7f, centroAnno 0b1f7dc, TRASH v1.2.  
**ArraySplitter pin:** commit d88578a (HEAD on origin/main). This commit chain includes: iter-2 SIMD-ED (01500ba); intermediate-HOR schema (718cf74, brief #5) which added `level` + `parent_idx` columns to `hors.tsv` and a `parent_level` column to `monomers.tsv`; `--max-period <usize>` CLI flag (d88578a, brief #4). Default `--max-period 5000` matches the iter-5 golden envelope. Golden MD5s in `state/scripts/regression_test/GOLDEN_MANIFEST.md`.

Cells with `#!bash` shebangs run on aglab0 via `ssh`; Python cells run locally and parse `summary.tsv` files into paper-cited numbers.

In [0]:
#!bash
# === Cell 1 - setup: write env vars used by every subsequent bash cell ===
cat > setup.env << 'ENVEOF'
SERVER=aglab0.utrail.org
BENCH_DIR=/home/akomissarov/asplit_bench
ZF_FASTA=/home/akomissarov/asplit_bench/zf.fa
HG_FASTA=/mnt/data/satelome/primates_t2t_final/all_fastas/satellome/hg002v1.1.mat/fastan/hg002v1.1.mat.10kb.fasta
ASPLIT_REPO=/home/akomissarov/asplit_bench/arraysplitter_repo   # canonical git clone, checked out at pinned SHA
ASPLIT_SHA=d88578a   # iter-5 chain: 01500ba iter-2 + 718cf74 intermediate-HORs + d88578a --max-period flag
ASPLIT_BIN=/home/akomissarov/asplit_bench/arraysplitter_repo/src/rust/arraysplitter/target/release/arraysplitter
FASTAN_BIN=/home/akomissarov/asplit_bench/FASTAN/bin/FasTAN
CENTROANNO_BIN=/home/akomissarov/asplit_bench/centroAnno/centroAnno
TRASH_RUN=/home/akomissarov/TRASH/TRASH_run.sh
OUT_DIR=/home/akomissarov/asplit_bench/notebook_out
ENVEOF
echo 'wrote setup.env'
ssh "$(grep '^SERVER=' setup.env | cut -d= -f2)" 'mkdir -p ~/asplit_bench/notebook_out && echo ok'

wrote setup.env
ok


## Phase 1 — Verify input panels exist on aglab0

Both panels are declared in `metadata.research_compiler.inputs[]` with MD5 pins. The viewer's Verify button cross-checks the on-disk MD5 against the declared one before any run.

In [0]:
#!bash
# === Cell 3 — verify panel MD5s on aglab0 ===
source ./setup.env
ssh "$SERVER" "
  echo '--- zfinch panel ---'
  md5sum '$ZF_FASTA'
  echo '--- HG002.mat panel ---'
  md5sum '$HG_FASTA'
"
echo
echo 'Expected (paper §3.1, §3.2):'
echo '  db4f049d73c4803bce6ae41ad7ffcef5  zf.fa            (60 Mb, 429 arrays)'
echo '  c97c33d38c9294131b89ab8f05ca3e11  hg002v1.1.mat.10kb.fasta (152 Mb, 1088 arrays)'

--- zfinch panel ---
db4f049d73c4803bce6ae41ad7ffcef5  /home/akomissarov/asplit_bench/zf.fa
--- HG002.mat panel ---
c97c33d38c9294131b89ab8f05ca3e11  /mnt/data/satelome/primates_t2t_final/all_fastas/satellome/hg002v1.1.mat/fastan/hg002v1.1.mat.10kb.fasta

Expected (paper §3.1, §3.2):
  db4f049d73c4803bce6ae41ad7ffcef5  zf.fa            (60 Mb, 429 arrays)
  c97c33d38c9294131b89ab8f05ca3e11  hg002v1.1.mat.10kb.fasta (152 Mb, 1088 arrays)


## Phase 2 - Build ArraySplitter

Build a single ArraySplitter binary at commit d88578a (iter-5, which landed brief #4 — the `--max-period <usize>` CLI flag, default 5000). All four cap variants in the §3.5 sweep are now reachable via the flag from one binary; no source patches required.

Built with `RUSTFLAGS='-C target-cpu=native'` for AVX2 + native-arch tuning. Default `--max-period 5000` is byte-identical to the iter-2 golden (01500ba) — the regression test in Phase 12 confirms 5/5 outputs match.

Earlier revisions of this notebook (pre-brief-#4) maintained four parallel `asplit_src{,_32000,_100000,_250000}` source trees as sed-patches of `decompose.rs:777`. That recipe is now obsolete and removed.

In [0]:
#!bash
# === Cell 5 - clone-or-fetch ArraySplitter at pinned SHA + build single binary ===
source ./setup.env
set -e
ssh "$SERVER" "
  set -e
  source ~/.cargo/env
  cd ~/asplit_bench
  if [ ! -d '$ASPLIT_REPO/.git' ]; then
    echo '--- cloning github.com/aglabx/ArraySplitter to \$(basename $ASPLIT_REPO) ---'
    rm -rf '$ASPLIT_REPO'
    git clone --quiet https://github.com/aglabx/ArraySplitter.git '$ASPLIT_REPO'
  else
    echo '--- repo exists; fetching latest ---'
    git -C '$ASPLIT_REPO' fetch --quiet
  fi
  git -C '$ASPLIT_REPO' checkout --quiet '$ASPLIT_SHA'
  echo '--- HEAD after checkout:'
  git -C '$ASPLIT_REPO' log -1 --oneline
  cd '$ASPLIT_REPO/src/rust/arraysplitter'
  echo '--- cargo build --release (RUSTFLAGS=target-cpu=native) ---'
  RUSTFLAGS='-C target-cpu=native' cargo build --release 2>&1 | tail -2
  echo '--- binary timestamp + --max-period flag assertion:'
  ls -lh '$ASPLIT_BIN'
  if '$ASPLIT_BIN' --help 2>&1 | grep -q -- '--max-period'; then
    echo 'OK: --max-period flag is present in --help (brief #4 + #5 chain confirmed)'
  else
    echo 'FAIL: --max-period missing in --help -> binary is stale or wrong SHA' >&2
    exit 1
  fi
"
echo
echo 'Default --max-period 5000 is byte-identical to iter-5 golden; Phase 12 regression test confirms.'

--- repo exists; fetching latest ---
--- HEAD after checkout:
d88578a Expose max_period autocorrelation cap as --max-period CLI flag
--- cargo build --release (RUSTFLAGS=target-cpu=native) ---
    Finished `release` profile [optimized] target(s) in 0.07s
--- binary timestamp + --max-period flag assertion:
-rwxrwxr-x 2 akomissarov akomissarov 1.9M May 23 17:24 /home/akomissarov/asplit_bench/arraysplitter_repo/src/rust/arraysplitter/target/release/arraysplitter
OK: --max-period flag is present in --help (brief #4 + #5 chain confirmed)

Default --max-period 5000 is byte-identical to iter-5 golden; Phase 12 regression test confirms.


## Phase 3 — Run ArraySplitter cap sweep (4 caps × 2 panels = 8 runs) — §3.1, §3.2, §3.5

Each binary processes each panel end-to-end (4 threads, AVX2). Wall-time and peak-RSS are read from ArraySplitter's own INFO log; the per-array `summary.tsv` is the source of truth for biology and cap-sweep tables.

**Expected wall-times** (paper §3.5 table, aglab0):

| cap | zfinch | HG002.mat |
|---:|---:|---:|
| 5 000 | 32.3 s | 192.7 s |
| 32 000 | 62.9 s | 240.0 s |
| 100 000 | 106.4 s | 313.9 s |
| 250 000 | 144.5 s | 417.7 s |

In [0]:
#!bash
# === Cell 7 - ArraySplitter cap sweep via --max-period flag (4 caps x 2 panels = 8 runs) ===
source ./setup.env
ssh "$SERVER" "
  mkdir -p '$OUT_DIR/asplit'
  for cap in 5000 32000 100000 250000; do
    for panel in zf hg; do
      if [ \"\$panel\" = 'zf' ]; then FASTA='$ZF_FASTA'; else FASTA='$HG_FASTA'; fi
      echo \"--- cap=\$cap panel=\$panel ---\"
      /usr/bin/time -p '$ASPLIT_BIN' \\
        -i \$FASTA -o '$OUT_DIR/asplit/'\${panel}_\${cap} \\
        -t 4 --method autocorr --max-period \$cap 2>&1 \\
        | grep -E 'in [0-9]|^real|memory|CPU' | head -3
    done
  done
"
echo
echo 'Outputs: ~/asplit_bench/notebook_out/asplit/{zf,hg}_{5000,32000,100000,250000}.{summary,hors,monomers,lengths,decomposed.fasta}.tsv'

--- cap=5000 panel=zf ---
2026-05-23 18:42:12 - arraysplitter - INFO - Processed 429 arrays, 60.01 Mbp in 33.313s
2026-05-23 18:42:12 - arraysplitter - INFO - Peak memory: 183.82 MB, avg CPU: 1076.6%
real 33.33
--- cap=5000 panel=hg ---


BashCancelled: cancelled by user

## Phase 4 — Run FasTAN on both panels — §3.1, §3.2

FasTAN (ad3002 fork, commit 963ac7f) is the fastest comparator. Single invocation per panel; output is `.1ano` per-array intervals + per-array `unit` size.

**Expected:** zfinch 1.54 s; HG002.mat 8.41 s.

In [ ]:
#!bash
# === Cell 9 - FasTAN on both panels ===
# Two FasTAN quirks worth knowing:
#   1. -oPATH and -T<N> must be glued (no space): '-o/path/zf' not '-o /path/zf';
#      '-T4' not '-T 4'. With a space FasTAN treats the next token as positional
#      input and prints help.
#   2. -m (.1ano mask) requires input as a GDB file (.1gdb), not raw FASTA.
#      Convert with FAtoGDB (from FASTGA) which writes <name>.1gdb next to <name>.fa.
source ./setup.env
ssh "$SERVER" "
  mkdir -p '$OUT_DIR/fastan'
  FAtoGDB=~/asplit_bench/FASTGA/FAtoGDB
  for panel in zf hg; do
    if [ \"\$panel\" = 'zf' ]; then FASTA='$ZF_FASTA'; else FASTA='$HG_FASTA'; fi
    GDB=\${FASTA%.*}.1gdb
    if [ ! -f \"\$GDB\" ]; then
      echo \"--- converting \$panel FASTA -> .1gdb via FAtoGDB ---\"
      \$FAtoGDB \$FASTA \$GDB
    else
      echo \"--- \$panel .1gdb already present: \$GDB\"
    fi
    OUT='$OUT_DIR/fastan/'\${panel}
    echo \"--- FasTAN -ampv -T4 -o\$OUT \$GDB ---\"
    /usr/bin/time -p '$FASTAN_BIN' -ampv -T4 -o\$OUT \$GDB 2>&1 | tail -10
    ls -lh \${OUT}.1ano 2>/dev/null || echo 'WARN: no .1ano produced'
  done
"
echo
echo 'Outputs: ~/asplit_bench/notebook_out/fastan/{zf,hg}.1ano'

## Phase 5 — Run centroAnno on both panels — §3.1, §3.2

centroAnno (commit 0b1f7dc) runs single-threaded in `anno-asm` mode. Cost on these panels is substantial — zfinch ≈ 56 min, HG002.mat ≈ 2 h 52 min. Coverage is partial: the `repCutoff = 0.2` filter rejects low-complexity arrays (42/429 on zfinch, 286/1088 on HG002).

**Expected wall-times** (paper §3.1, §3.2): zfinch 3 346.7 s (≈ 56 min); HG002.mat 10 328 s (≈ 2 h 52 min).

In [ ]:
#!bash
# === Cell 11 — centroAnno on both panels (LONG — kick off in background, monitor separately) ===
source ./setup.env
ssh "$SERVER" "
  mkdir -p '$OUT_DIR/centro/zf' '$OUT_DIR/centro/hg'
  cd ~/asplit_bench/centroAnno
  echo '--- centroAnno on zfinch (≈ 56 min) ---'
  nohup /usr/bin/time -p ./centroAnno '$ZF_FASTA' -o '$OUT_DIR/centro/zf/zf' -x anno-asm \
    > '$OUT_DIR/centro/zf/centroAnno_zf.log' 2>&1 &
  echo \"   pid=\$!\"
  echo '--- centroAnno on HG002.mat (≈ 2 h 52 min) ---'
  nohup /usr/bin/time -p ./centroAnno '$HG_FASTA' -o '$OUT_DIR/centro/hg/hg' -x anno-asm \
    > '$OUT_DIR/centro/hg/centroAnno_hg.log' 2>&1 &
  echo \"   pid=\$!\"
"
echo
echo 'Both centroAnno runs kicked off in background. Monitor with:'
echo '  ssh aglab0.utrail.org "tail -5 ~/asplit_bench/notebook_out/centro/zf/centroAnno_zf.log"'
echo '  ssh aglab0.utrail.org "ls ~/asplit_bench/notebook_out/centro/{zf,hg}/{zf,hg}/*_decomposedResult.csv | wc -l"'

## Phase 6 — Run TRASH on a representative subset — §3.1 footnote

TRASH v1.2 (Wlodzimierz et al., 2023) targets in-depth single-array analysis; observed cost is ≈ 15–20 min per Mb-scale array. Full-panel runs are therefore not feasible (extrapolated 5+ days on the zfinch panel; longer on HG002). We sample five representative arrays covering the principal architectural classes: a 6 bp microsatellite, a 191 bp avian α-satellite, a 696 bp composite monomer class, a chrW 4 632 bp HOR, and the chr17 98 714 bp human HOR.

TRASH conda environment is `trash` with R 4.6 + bioconductor-biostrings + pwalign + circlize + seqinr (install steps captured in [[BTN-trash-r-deps]]).

In [ ]:
#!bash
# === Cell 13 — build representative subset FASTA + run TRASH on each ===
source ./setup.env
ssh "$SERVER" "
  mkdir -p '$OUT_DIR/trash_subset'
  # Extract 5 representative arrays into one small FASTA. Names taken from paper §3.1.
  python3 - << 'PY'
from Bio import SeqIO
wanted = {
  'zf_microsat_6bp':   ('$ZF_FASTA', 'chr1A_mat_59204945_59239682'),
  'zf_alpha_191bp':    ('$ZF_FASTA', 'chr1A_pat_72241238_72348515'),
  'zf_composite_700bp':('$ZF_FASTA', 'chr1_mat_20177056_20377500'),
  'zf_chrW_4632bp':    ('$ZF_FASTA', 'chrW_pat_'),                  # prefix match
  'hg_chr17_98714bp':  ('$HG_FASTA', 'chr17_MATERNAL_43624480'),
}
out = open('$OUT_DIR/trash_subset/subset.fa','w')
for tag, (src, key) in wanted.items():
  for rec in SeqIO.parse(src, 'fasta'):
    if rec.id.startswith(key):
      out.write(f'>{tag}__{rec.id}\n{str(rec.seq)}\n'); break
out.close()
PY
  ls -lh '$OUT_DIR/trash_subset/subset.fa'
  source ~/miniconda3/etc/profile.d/conda.sh && conda activate trash
  rm -rf '$OUT_DIR/trash_subset/out'
  /usr/bin/time -p '$TRASH_RUN' --def '$OUT_DIR/trash_subset/subset.fa' --o '$OUT_DIR/trash_subset/out' 2>&1 | tail -6
  ls -lh '$OUT_DIR/trash_subset/out/subset.fa_out/' | head -15
"
echo
echo 'Representative TRASH outputs land in ~/asplit_bench/notebook_out/trash_subset/out/subset.fa_out/'

## Phase 7 — Pull all summary.tsv files locally for parsing

Once Phases 3–5 are done, rsync the per-tool summaries to `scratch/` for local Python parsing. Output files are gitignored under `outputs/notebooks/<nb>/out/`.

In [ ]:
#!bash
# === Cell 15 — pull ArraySplitter summary.tsv files + FasTAN .1ano files locally ===
source ./setup.env
mkdir -p out
rsync -az "$SERVER:$OUT_DIR/asplit/" out/asplit/
rsync -az "$SERVER:$OUT_DIR/fastan/" out/fastan/
ls -lh out/asplit/*.summary.tsv
ls -lh out/fastan/

## Phase 8 — §3.5 cap-sweep cost-benefit table (Python parse)

For each of the 8 ArraySplitter runs (4 caps × 2 panels), count:

- total arrays processed
- arrays with detected HOR (`hor_period ≥ 1.5 × mono_period`)
- arrays with `hor_period > 5 000` (impossible at default cap)
- max detected `hor_period`
- arrays where `hor_period` changed vs cap = 5000 (cap raised the answer)

The output matches the §3.5 table verbatim.

In [ ]:
import csv, pathlib, collections

def load(panel, cap):
    p = pathlib.Path(f'out/asplit/{panel}_{cap}.summary.tsv')
    if not p.exists():
        print(f'  MISSING: {p}'); return None
    return list(csv.DictReader(p.open(), delimiter='\t'))

rows = []
for panel in ('zf', 'hg'):
    base = {r['array_id']: r for r in (load(panel, 5000) or [])}
    for cap in (5000, 32000, 100000, 250000):
        rs = load(panel, cap)
        if rs is None: continue
        n = len(rs)
        hp = [int(r['hor_period']) for r in rs if r['hor_period'].isdigit() and int(r['hor_period']) > 0]
        real_hor = sum(1 for r in rs
                       if r['hor_period'].isdigit() and r['mono_period'].isdigit()
                       and int(r['hor_period']) >= 1.5 * int(r['mono_period']) and int(r['mono_period']) > 0)
        big = sum(1 for r in rs if r['hor_period'].isdigit() and int(r['hor_period']) > 5000)
        if cap == 5000:
            changed = 0
        else:
            changed = sum(1 for r in rs
                          if r['array_id'] in base
                          and r['hor_period'].isdigit() and base[r['array_id']]['hor_period'].isdigit()
                          and int(r['hor_period']) != int(base[r['array_id']]['hor_period']))
        rows.append((panel, cap, n, max(hp) if hp else 0, real_hor, big, changed))

print(f"{'panel':<6} {'cap':>7} {'n_arr':>6} {'max_hp':>8} {'real_HOR':>9} {'hp>5K':>7} {'Δ vs 5K':>9}")
for row in rows:
    print(f'{row[0]:<6} {row[1]:>7} {row[2]:>6} {row[3]:>8} {row[4]:>9} {row[5]:>7} {row[6]:>9}')

## Phase 9 — §3.3 biology: three principal monomer classes + 88 HOR arrays + 191 → 13 bp

Parse `zf_5000.summary.tsv` (the default-cap reference panel) into the three numbers cited in §3.3:

1. Three principal monomer classes by `mono_period` bucket (≈ 6 bp microsatellite, 191 bp avian α-satellite-like, 696–717 bp composite)
2. 88/429 arrays with real HOR (`hor_period ≥ 1.5 × mono_period`)
3. The 8 chromosomes where 191 bp HOR decomposes into 10–13 bp microsatellite sub-units

In [ ]:
import csv, pathlib, collections

rs = list(csv.DictReader(pathlib.Path('out/asplit/zf_5000.summary.tsv').open(), delimiter='\t'))
n = len(rs)
print(f'Total zfinch arrays in default-cap panel: {n}')

# Three principal monomer classes
classes = collections.Counter()
for r in rs:
    if not r['mono_period'].isdigit(): continue
    mp = int(r['mono_period'])
    if mp < 10:           classes['~6 bp microsat']         += 1
    elif 180 <= mp <= 200: classes['191 bp avian α-sat']     += 1
    elif 680 <= mp <= 720: classes['696–717 bp composite']   += 1
    elif 800 <= mp <= 3000:classes['800–3000 bp long-composite'] += 1
    else:                  classes[f'other ({mp})']           += 0
print()
print('Three principal monomer classes (§3.3 table):')
for label, count in classes.most_common():
    print(f'  {label:<30s} {count:>4d}  ({100*count/n:5.1f}%)')

# Real HOR arrays
real_hor = [r for r in rs
            if r['hor_period'].isdigit() and r['mono_period'].isdigit()
            and int(r['hor_period']) >= 1.5 * int(r['mono_period']) and int(r['mono_period']) > 0]
print()
print(f'Arrays with HOR (hor ≥ 1.5 × mono): {len(real_hor)} / {n}  ({100*len(real_hor)/n:.1f}%)')

# 191 bp HOR built from 10-13 bp sub-units
matryoshka_191 = [r for r in real_hor
                  if 180 <= int(r['hor_period']) <= 200
                  and 8 <= int(r['mono_period']) <= 15]
chroms_191 = sorted({r['array_id'].split('_')[0] for r in matryoshka_191})
print()
print(f'191 bp HOR → 10–13 bp microsatellite sub-units: {len(matryoshka_191)} arrays')
print(f'  chromosomes: {chroms_191}')

# chrW extreme HOR
chrW = [r for r in real_hor if r['array_id'].startswith('chrW')]
for r in chrW:
    hp, mp = int(r['hor_period']), int(r['mono_period'])
    print(f'  chrW HOR: {r["array_id"]}  hor={hp} bp / mono={mp} bp = {hp/mp:.1f}x')

## Phase 10 - sec3.6 nested HOR matryoshka via intermediate-HOR rows in hors.tsv

Since brief #5 (718cf74), `hors.tsv` carries first-class rows for every intermediate HOR level detected by the recursive descent. Each row has `level` (1 = top-level HOR, 2+ = sub-HORs from recursion) and `parent_idx` (idx of the parent at level-1, or -1 at level 1). The previous flat `period_classes` histogram in `summary.tsv` is now redundant.

**Expected on the zfinch panel** (from the brief #5 landing report, default cap=5000):

| level | sub_hor rows | Notes |
|---:|---:|---|
| 1 (top) | 429 | one per array (already exists pre-brief-#5) |
| 2 | 6 076 | newly visible |
| 3 | 2 329 | newly visible |
| 4 | 246 | newly visible |
| 5 | 11 | newly visible — deepest matryoshka on the panel |

Total `hors.tsv` rows: 435 684 (iter-2) -> 444 346 (iter-5) = +8 662 sub_hor rows.  
Heaviest single array: `chr34_pat_4952764_5760300` with 1 105 sub_hor rows.

In [ ]:
import csv, pathlib, collections

hors = list(csv.DictReader(pathlib.Path('out/asplit/zf_5000.hors.tsv').open(), delimiter='\t'))

# Level distribution
by_level = collections.Counter(r.get('level', '?') for r in hors)
print('hors.tsv level distribution (zfinch, default cap=5000):')
for level in sorted(by_level, key=lambda x: (x == '?', int(x) if x.isdigit() else 999)):
    print(f'  level={level:>3}: {by_level[level]:>6} rows')

# Max recursion depth on the panel
max_depth = max((int(r['level']) for r in hors if r.get('level','').isdigit()), default=0)
print(f'\nMax recursion depth observed: {max_depth}')

# Per-array sub_hor counts
by_array = collections.Counter()
for r in hors:
    if r.get('level','').isdigit() and int(r['level']) >= 2:
        by_array[r['array_id']] += 1
print(f'\nTop 5 arrays by sub_hor count (deepest matryoshka):')
for aid, n in by_array.most_common(5):
    print(f'  {aid:<55} {n:>5} sub_hor rows')

# Walk the tree for ONE specific array as a worked example
target = 'chr1A_mat_59204945_59239682'
tree = [r for r in hors if r['array_id'] == target]
print(f'\nWorked example - full matryoshka for {target}:')
for r in sorted(tree, key=lambda x: (int(x['level']) if x.get('level','').isdigit() else 0,
                                     int(x['idx']) if x.get('idx','').isdigit() else 0)):
    lvl = r.get('level', '?')
    parent = r.get('parent_idx', '?')
    period = r.get('period', '?')
    print(f'  level={lvl:>2}  idx={r["idx"]:>4}  parent_idx={parent:>4}  period={period:>6}')

## Phase 11 — §3.1 / §3.4 period agreement + period-aware cut Jaccard (ArraySplitter ↔ FasTAN)

Parse FasTAN's `.1ano` output via FASTGA's `ONEview`, project to per-array cut positions, then compute:

- **Period agreement** — fraction of arrays where ArraySplitter `hor_period` matches FasTAN `unit` within ±5 bp or ±5 % (paper: 83.4 %)
- **Period-aware cut Jaccard** — fold each cut delta modulo the per-array detected period, tally exact / ±10 bp / ±100 bp agreement (paper: 85.6 % / 95.2 % / 99.5 %)

The parser script lives at `state/scripts/smoke_asplit_vs_fastan/compare_cuts_zfinch.py`.

In [ ]:
#!bash
# === Cell 21 — period agreement + period-aware Jaccard (re-uses existing parser) ===
PARSER=../../state/scripts/smoke_asplit_vs_fastan/compare_cuts_zfinch.py
if [ ! -f "$PARSER" ]; then
  echo "parser not found at $PARSER — run from notebook directory"
  exit 1
fi
python3 "$PARSER" \
    --asplit-summary out/asplit/zf_5000.summary.tsv \
    --asplit-hors    out/asplit/zf_5000.hors.tsv \
    --fastan-1ano    out/fastan/zf.1ano \
    --tolerance      10
echo
echo 'Expected (paper §3.1): 83.4% period agreement; 85.6%/95.2%/99.5% cut agreement at ±0/±10/±100 bp.'

## Phase 9b - sec3.3 expanded: four HOR-architecture tiers

The sec3.3 'most striking architectures' table has four rows:

| Group | Arrays | HOR period | Monomer period | Ratio |
|---|---:|---:|---:|---:|
| 191 bp HOR built from 10-13 bp microsats (chr16/34/35) | 8 | 191 bp | 10-13 bp | 17-19x |
| 2.5-4 kb HOR from ~190 bp alpha-sat-like (chr1/3/5) | 6 | 2783-4032 bp | 190-207 bp | 14-20x |
| Medium 2-2.8 kb composite HORs (chr1/5/Z) | 4 | 2107-2817 bp | 162-207 bp | 13-17x |
| chrW extreme HOR | 1 | 4632 bp | 98 bp | 47x |

This cell extracts all four tiers from zf_5000.summary.tsv.

In [ ]:
import csv, pathlib

rs = list(csv.DictReader(pathlib.Path('out/asplit/zf_5000.summary.tsv').open(), delimiter='\t'))

def parse(r):
    try:
        return int(r['hor_period']), int(r['mono_period']), r['array_id']
    except:
        return None

real_hor = [t for t in (parse(r) for r in rs) if t is not None
            and t[0] >= 1.5 * t[1] and t[1] > 0]

tiers = {
    'Tier 1 - 191 bp HOR from 10-13 bp microsats':
        lambda hp, mp, _: 180 <= hp <= 200 and 8 <= mp <= 15,
    'Tier 2 - 2.5-4 kb HOR from ~190 bp alpha-sat':
        lambda hp, mp, _: 2500 <= hp <= 4200 and 180 <= mp <= 220,
    'Tier 3 - medium 2-2.8 kb composite HORs':
        lambda hp, mp, _: 2000 <= hp <= 2900 and 150 <= mp <= 220,
    'Tier 4 - chrW extreme HOR (4632 / 98 bp)':
        lambda hp, mp, aid: aid.startswith('chrW') and hp >= 4000,
}

for tier_name, predicate in tiers.items():
    matches = [(hp, mp, aid) for hp, mp, aid in real_hor if predicate(hp, mp, aid)]
    print(f'{tier_name}: {len(matches)} arrays')
    for hp, mp, aid in sorted(matches, key=lambda x: -x[0]/x[1])[:6]:
        print(f'    {aid:<55} hor={hp} bp / mono={mp} bp = {hp/mp:.1f}x')
    print()

## Phase 11b - optimisation history (319 s -> 28.5 s = 11.2x speedup)

The paper sec3 (and the methodology preprint in preparation) cites a cumulative 11.2x speedup from the 2-iteration SIMD-edit-distance optimisation:

| Iteration | Commit | Wall on zfinch panel | Speedup vs baseline |
|---|---|---:|---:|
| baseline (pre-opt) | parent of e23932c | 319 s | 1.00x |
| iter-1 (triple_accel::levenshtein_exp in main path) | 4b4f7da | 213 s | 1.50x |
| iter-2 (k-bounded levenshtein_simd_k in multiplet split) | 01500ba | 28.5 s | 11.2x |

The numbers come from hyperfine runs landed in EXP-asplit-opt-iter1-triple-accel and EXP-asplit-opt-iter2-bounded-simd-ed. For compactness, we re-verify only that the current binary still matches iter-2 timing on the same panel via hyperfine.

In [ ]:
#!bash
# === Verify current binary still hits iter-2 timing on zfinch ===
source ./setup.env
ssh "$SERVER" "
  echo '--- hyperfine: 3 warmup + 5 runs of iter-2 ArraySplitter on zfinch ---'
  command -v hyperfine >/dev/null || { echo 'hyperfine not installed; skipping'; exit 0; }
  cd ~/asplit_bench
  hyperfine --warmup 3 --runs 5 \
    '$ASPLIT_BIN -i $ZF_FASTA -o $OUT_DIR/regression_check -t 4 --method autocorr' \
    2>&1 | tail -8
"
echo
echo 'Expected: median wall around 28.5 s on AVX2 host or ~32 s on aglab0 96-core (rayon spawn churn).'

## Phase 12 — regression-test ArraySplitter against golden MD5s

Final reproducibility check: the v1.7.4 commit-01500ba ArraySplitter binary must produce byte-identical outputs for the zfinch panel, as locked in `state/scripts/regression_test/GOLDEN_MANIFEST.md`. This guards against silent regressions in the SIMD-ED optimisation path.

In [ ]:
#!bash
# === Cell 23 — regression test ===
REG=../../state/scripts/regression_test/run_regression.sh
if [ -x "$REG" ]; then
  bash "$REG"
else
  echo "$REG not found or not executable; see GOLDEN_MANIFEST.md for the expected MD5s:"
  cat ../../state/scripts/regression_test/GOLDEN_MANIFEST.md 2>/dev/null | head -15
fi

## Reproduction summary

If every cell above ran cleanly, every numerical claim in `paper-draft-2026-05-21-v2.md` sections 3.1-3.6 is sourced from a pinned-MD5 input and a deterministic ArraySplitter binary at `d88578a`.

**Open items (numbers not yet auto-checked here):**

- TRASH per-array timing on the full panel (out of scope; representative subset only).
- centroAnno period agreement vs ArraySplitter on zfinch (parser exists for FasTAN only; centroAnno parser pending).